# Daily revenue features
Productionized as a scheduled papermill job:
`papermill revenue_daily.ipynb /tmp/out.ipynb -p as_of 2024-03-01`

In [1]:
as_of = "2024-03-01"
orders_path = "s3://analytics-prod/orders/"
output_path = "s3://analytics-prod/features/revenue/"

In [2]:
import pandas as pd
import numpy as np

orders = pd.read_parquet(orders_path)
orders["ordered_at"] = pd.to_datetime(orders["ordered_at"], utc=True)

In [3]:
as_of_ts = pd.Timestamp(as_of, tz="UTC")
day_start = as_of_ts - pd.Timedelta(days=1)
orders = orders[(orders["ordered_at"] >= day_start) & (orders["ordered_at"] < as_of_ts)]
orders["amount"] = orders["amount"].astype("float64")

In [9]:
orders[["amount"]].describe()
orders["region"].value_counts(dropna=False)

In [4]:
daily = (
    orders.groupby("customer_id")
    .agg(
        revenue=("amount", "sum"),
        order_count=("amount", "size"),
        avg_order_value=("amount", "mean"),
    )
    .reset_index()
)

In [5]:
daily["as_of"] = as_of
daily.to_parquet(output_path + f"as_of={as_of}/features.parquet", index=False)